# LeetCode 2121 — Intervals Between Identical Elements bằng Python

Link đề: https://leetcode.com/problems/intervals-between-identical-elements/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def getDistances(self, arr: List[int]) -> List[int]:
        ...
```

(Đề này giống hệt LC 2615 *Sum of Distances*.)

## 1. Mục tiêu học

1. Gom chỉ số theo giá trị bằng `defaultdict(list)` — kỹ thuật "nhóm" cực kỳ phổ biến.
2. Biến tổng khoảng cách `Σ |p − q|` thành công thức **tổng tiền tố**: tách phần bên trái và phần bên phải.
3. Hiểu **công thức truy hồi** (rolling): khi dịch từ chỉ số `p_{j−1}` sang `p_j`, tổng thay đổi bao nhiêu.
4. Viết lời giải **hai lượt quét trái/phải** với hai `dict` (đếm, tổng) — không cần gom nhóm.
5. Đối chiếu với brute force O(n²) trên input nhỏ và đo thời gian trên input lớn.

## 2. Đề bài & ví dụ

Cho mảng `arr` gồm `n` số nguyên (chỉ số từ 0). **Khoảng cách** giữa `arr[i]` và `arr[j]` là `|i − j|`.

Trả về mảng `intervals` độ dài `n`, với `intervals[i]` = **tổng khoảng cách** từ `i` tới **mọi** chỉ số `j` có `arr[j] == arr[i]` (kể cả `j = i`, nhưng `|i − i| = 0` nên không ảnh hưởng).

**Ví dụ 1:** `arr = [2,1,3,1,2,3,3]` → `[4,2,7,2,4,4,5]`.
- Chỉ số 2 (giá trị 3): các số 3 khác ở 5 và 6 ⇒ `|2−5| + |2−6| = 7`.
- Chỉ số 6 (giá trị 3): `|6−2| + |6−5| = 5`.

**Ví dụ 2:** `arr = [10,5,10,10]` → `[5,0,3,4]` (số 5 xuất hiện một lần ⇒ 0).

**Ràng buộc:** `n == arr.length`, `1 ≤ n ≤ 10⁵`, `1 ≤ arr[i] ≤ 10⁵`.

In [1]:
from typing import List
from collections import defaultdict
from itertools import accumulate
import random, time

examples = [
    ([2, 1, 3, 1, 2, 3, 3], [4, 2, 7, 2, 4, 4, 5]),
    ([10, 5, 10, 10], [5, 0, 3, 4]),
]
for arr, exp in examples:
    print("arr =", arr, "-> kỳ vọng", exp)
    for i, v in enumerate(arr):
        same = [j for j, w in enumerate(arr) if w == v and j != i]
        print(f"   i={i} (giá trị {v}): cùng giá trị ở {same} -> " +
              " + ".join(f"|{i}-{j}|" for j in same) + f" = {sum(abs(i-j) for j in same)}")

arr = [2, 1, 3, 1, 2, 3, 3] -> kỳ vọng [4, 2, 7, 2, 4, 4, 5]
   i=0 (giá trị 2): cùng giá trị ở [4] -> |0-4| = 4
   i=1 (giá trị 1): cùng giá trị ở [3] -> |1-3| = 2
   i=2 (giá trị 3): cùng giá trị ở [5, 6] -> |2-5| + |2-6| = 7
   i=3 (giá trị 1): cùng giá trị ở [1] -> |3-1| = 2
   i=4 (giá trị 2): cùng giá trị ở [0] -> |4-0| = 4
   i=5 (giá trị 3): cùng giá trị ở [2, 6] -> |5-2| + |5-6| = 4
   i=6 (giá trị 3): cùng giá trị ở [2, 5] -> |6-2| + |6-5| = 5
arr = [10, 5, 10, 10] -> kỳ vọng [5, 0, 3, 4]
   i=0 (giá trị 10): cùng giá trị ở [2, 3] -> |0-2| + |0-3| = 5
   i=1 (giá trị 5): cùng giá trị ở [] ->  = 0
   i=2 (giá trị 10): cùng giá trị ở [0, 3] -> |2-0| + |2-3| = 3
   i=3 (giá trị 10): cùng giá trị ở [0, 2] -> |3-0| + |3-2| = 4


## 3. Kiến thức nền

### 3.1 Gom nhóm chỉ số theo giá trị

`defaultdict(list)` + `enumerate`: duyệt một lượt, mỗi giá trị có danh sách chỉ số **tăng dần** (vì ta duyệt từ trái sang phải).

In [2]:
arr = [2, 1, 3, 1, 2, 3, 3]
groups = defaultdict(list)
for i, v in enumerate(arr):
    groups[v].append(i)
print(dict(groups))

{2: [0, 4], 1: [1, 3], 3: [2, 5, 6]}


### 3.2 Tổng tiền tố với `itertools.accumulate`

`pre[k]` = tổng `k` phần tử đầu. Tổng đoạn `[l, r)` = `pre[r] − pre[l]`. Thêm `initial=0` để `pre[0] = 0`.

In [3]:
p = [2, 5, 6]
pre = list(accumulate(p, initial=0))
print("pre =", pre)
print("tổng p[1:3] =", pre[3] - pre[1])

pre = [0, 2, 7, 13]
tổng p[1:3] = 11


### 3.3 Bỏ dấu giá trị tuyệt đối khi đã sắp thứ tự

Nếu `q < p` thì `|p − q| = p − q`; nếu `q > p` thì `|p − q| = q − p`. Khi danh sách chỉ số đã **tăng dần**, ta biết chính xác ai bên trái, ai bên phải — không cần `abs` nữa:

`Σ_{q} |p − q| = Σ_{q<p} (p − q) + Σ_{q>p} (q − p) = p·L − (tổng bên trái) + (tổng bên phải) − p·R`

với `L`, `R` là số phần tử bên trái/phải.

In [4]:
p_list = [2, 5, 6]
j = 1; p = p_list[j]
L, R = j, len(p_list) - j - 1
left_sum, right_sum = sum(p_list[:j]), sum(p_list[j+1:])
print("công thức:", p * L - left_sum + right_sum - p * R)
print("trực tiếp:", sum(abs(p - q) for q in p_list))

công thức: 4
trực tiếp: 4


### 3.4 Python int không tràn số

Đáp án có thể lớn: một giá trị lặp `10⁵` lần ⇒ tổng khoảng cách tại đầu mút ≈ `Σ_{d=1}^{10⁵} d ≈ 5·10⁹` — vượt `int32` (Java/C++ phải dùng `long`). Python tự động dùng số nguyên lớn nên không lo.

In [5]:
n = 10**5
print("tổng khoảng cách từ chỉ số 0 khi mọi phần tử bằng nhau:", n * (n - 1) // 2, "> 2^31 - 1 =", 2**31 - 1)

tổng khoảng cách từ chỉ số 0 khi mọi phần tử bằng nhau: 4999950000 > 2^31 - 1 = 2147483647


## 4. Ý tưởng nền tảng / trực giác

Mỗi giá trị **độc lập** với các giá trị khác ⇒ chỉ cần giải bài con: cho danh sách chỉ số tăng dần `p_0 < p_1 < … < p_{k−1}`, tính `S_j = Σ_t |p_j − p_t|` cho mọi `j`.

Làm tay với giá trị 3 ở ví dụ 1: `p = [2, 5, 6]`.

| j | p_j | bên trái | bên phải | S_j |
|---|---|---|---|---|
| 0 | 2 | — | 5, 6 | (5−2)+(6−2) = 7 |
| 1 | 5 | 2 | 6 | (5−2)+(6−5) = 4 |
| 2 | 6 | 2, 5 | — | (6−2)+(6−5) = 5 |

**Quan sát truy hồi:** từ `j = 0` sang `j = 1`, ta dịch điểm đang xét sang phải một đoạn `Δ = 5 − 2 = 3`.
- `j = 1` phần tử bên trái (ở đây là chỉ số 2, cùng với chính điểm cũ) *xa thêm* `Δ` mỗi cái ⇒ `+1·Δ`.
- `k − 1 = 2` phần tử còn lại (5 và 6) *gần lại* `Δ` mỗi cái ⇒ `−2·Δ`.
- `S_1 = S_0 + 1·3 − 2·3 = 7 − 3 = 4`. ✔
- Sang `j = 2`: `Δ = 1`, `S_2 = 4 + 2·1 − 1·1 = 5`. ✔

In [6]:
p = [2, 5, 6]; k = len(p)
S = sum(q - p[0] for q in p)
print(f"S_0 = {S}")
for j in range(1, k):
    d = p[j] - p[j - 1]
    S += j * d - (k - j) * d
    print(f"j={j}: Δ={d}, có {j} điểm bên trái (xa thêm), {k-j} điểm còn lại (gần lại) -> S_{j} = {S}")

S_0 = 7
j=1: Δ=3, có 1 điểm bên trái (xa thêm), 2 điểm còn lại (gần lại) -> S_1 = 4
j=2: Δ=1, có 2 điểm bên trái (xa thêm), 1 điểm còn lại (gần lại) -> S_2 = 5


## 5. Cách 1 — Brute Force / baseline: duyệt mọi cặp

**Ý tưởng:** với mỗi `i`, duyệt mọi `j`, nếu `arr[j] == arr[i]` thì cộng `|i − j|`.

In [7]:
def brute(arr):
    n = len(arr)
    ans = [0] * n
    for i in range(n):
        for j in range(n):
            if arr[i] == arr[j]:
                ans[i] += abs(i - j)
    return ans

for arr, exp in examples:
    got = brute(arr)
    print(got, "OK" if got == exp else "SAI")

[4, 2, 7, 2, 4, 4, 5] OK
[5, 0, 3, 4] OK


**Độ phức tạp:** O(n²) thời gian, O(n) bộ nhớ. Với `n = 10⁵` là 10¹⁰ phép so sánh ⇒ quá chậm.

### 5.1 Baseline cải tiến nhẹ: gom nhóm rồi duyệt cặp trong nhóm

Chỉ so sánh trong cùng nhóm ⇒ O(Σ k_v²). Nhanh hơn khi giá trị đa dạng, nhưng **worst-case vẫn O(n²)** khi mọi phần tử bằng nhau.

**Nút thắt:** với mỗi `p_j` ta cộng lại từ đầu `k` khoảng cách, trong khi `S_j` và `S_{j+1}` chia sẻ gần hết thông tin.

In [8]:
def group_pairs(arr):
    groups = defaultdict(list)
    for i, v in enumerate(arr):
        groups[v].append(i)
    ans = [0] * len(arr)
    for p in groups.values():
        for a in p:
            ans[a] = sum(abs(a - b) for b in p)
    return ans

for arr, exp in examples:
    print(group_pairs(arr), "OK" if group_pairs(arr) == exp else "SAI")
for n in [1000, 4000]:
    a = [7] * n
    t = time.perf_counter(); group_pairs(a); print(f"n={n} (mọi phần tử bằng nhau): {(time.perf_counter()-t)*1000:.0f} ms")

[4, 2, 7, 2, 4, 4, 5] OK
[5, 0, 3, 4] OK
n=1000 (mọi phần tử bằng nhau): 36 ms


n=4000 (mọi phần tử bằng nhau): 565 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** trong một nhóm đã sắp tăng `p_0 < … < p_{k−1}`, đặt `pre[t] = p_0 + … + p_{t−1}`. Khi đó:

`S_j = p_j · j − pre[j]  +  (pre[k] − pre[j+1]) − p_j · (k − j − 1)`

- Phần đầu: `j` phần tử bên trái, mỗi phần tử đóng góp `p_j − p_t`.
- Phần sau: `k − j − 1` phần tử bên phải, mỗi phần tử đóng góp `p_t − p_j`.

Mỗi `S_j` tính O(1) ⇒ cả nhóm O(k) ⇒ tổng O(n).

**Quy tắc "nếu… thì…":**
- **Nếu** cần tổng khoảng cách tới một tập điểm trên đường thẳng **thì** sắp xếp và tách trái/phải để bỏ `abs`.
- **Nếu** tập điểm thay đổi dần (quét trái→phải) **thì** chỉ cần duy trì **số lượng** và **tổng** các điểm đã gặp: đóng góp bên trái của `i` = `cnt·i − sum`.

**Insight 2 (không cần gom nhóm):** quét trái→phải, giữ `cnt[v]` và `sm[v]` = số lần và tổng chỉ số các lần gặp `v` *trước* `i` ⇒ `left[i] = cnt[v]·i − sm[v]`. Quét phải→trái tương tự cho `right[i] = sm'[v] − cnt'[v]·i`. Đáp án = `left + right`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Gom nhóm + tổng tiền tố

Ba bước: (1) gom nhóm, (2) với mỗi nhóm dựng `pre`, (3) áp công thức.

In [9]:
def group_prefix(arr):
    groups = defaultdict(list)
    for i, v in enumerate(arr):
        groups[v].append(i)
    ans = [0] * len(arr)
    for p in groups.values():
        k = len(p)
        pre = list(accumulate(p, initial=0))
        for j, pj in enumerate(p):
            left = pj * j - pre[j]
            right = (pre[k] - pre[j + 1]) - pj * (k - j - 1)
            ans[pj] = left + right
    return ans

for arr, exp in examples:
    got = group_prefix(arr)
    print(got, "OK" if got == exp else "SAI")

[4, 2, 7, 2, 4, 4, 5] OK
[5, 0, 3, 4] OK


Code nộp LeetCode (cách 2):

In [10]:
class SolutionPrefix:
    def getDistances(self, arr: List[int]) -> List[int]:
        groups = defaultdict(list)
        for i, v in enumerate(arr):
            groups[v].append(i)
        ans = [0] * len(arr)
        for p in groups.values():
            k = len(p)
            total = sum(p)
            run = 0                       # pre[j] tích lũy dần, khỏi tạo list
            for j, pj in enumerate(p):
                ans[pj] = pj * j - run + (total - run - pj) - pj * (k - j - 1)
                run += pj
        return ans

print(SolutionPrefix().getDistances([2, 1, 3, 1, 2, 3, 3]))

[4, 2, 7, 2, 4, 4, 5]


**Độ phức tạp cách 2:** O(n) thời gian (mỗi chỉ số thuộc đúng một nhóm), O(n) bộ nhớ.

### 7.2 Cách 3 — Công thức truy hồi (rolling) trong nhóm

Từ mục 4: `S_0 = Σ p_t − k·p_0`, rồi `S_j = S_{j−1} + j·Δ − (k − j)·Δ` với `Δ = p_j − p_{j−1}`.

Giải thích: khi điểm xét dịch từ `p_{j−1}` sang `p_j`, có `j` điểm (`p_0..p_{j−1}`) nằm bên trái hoặc trùng điểm cũ — mỗi điểm xa thêm `Δ`; `k − j` điểm còn lại (`p_j..p_{k−1}`) đều ở bên phải điểm cũ và không bên trái điểm mới — mỗi điểm gần lại `Δ`.

In [11]:
def group_rolling(arr):
    groups = defaultdict(list)
    for i, v in enumerate(arr):
        groups[v].append(i)
    ans = [0] * len(arr)
    for p in groups.values():
        k = len(p)
        S = sum(p) - k * p[0]
        ans[p[0]] = S
        for j in range(1, k):
            d = p[j] - p[j - 1]
            S += j * d - (k - j) * d
            ans[p[j]] = S
    return ans

for arr, exp in examples:
    got = group_rolling(arr)
    print(got, "OK" if got == exp else "SAI")

[4, 2, 7, 2, 4, 4, 5] OK
[5, 0, 3, 4] OK


**Độ phức tạp cách 3:** O(n) thời gian, O(n) bộ nhớ. Ưu điểm: không cần mảng tiền tố, chỉ một biến `S`.

### 7.3 Cách 4 — Hai lượt quét với `dict` đếm/tổng (không gom nhóm)

- Lượt trái→phải: `left[i] = cnt[v]·i − sm[v]`, rồi cập nhật `cnt[v] += 1`, `sm[v] += i`.
- Lượt phải→trái: `right[i] = sm[v] − cnt[v]·i` (các chỉ số đã gặp đều lớn hơn `i`), rồi cập nhật.

In [12]:
def two_pass(arr):
    n = len(arr)
    ans = [0] * n
    cnt, sm = defaultdict(int), defaultdict(int)
    for i, v in enumerate(arr):
        ans[i] += cnt[v] * i - sm[v]
        cnt[v] += 1; sm[v] += i
    cnt.clear(); sm.clear()
    for i in range(n - 1, -1, -1):
        v = arr[i]
        ans[i] += sm[v] - cnt[v] * i
        cnt[v] += 1; sm[v] += i
    return ans

for arr, exp in examples:
    got = two_pass(arr)
    print(got, "OK" if got == exp else "SAI")

[4, 2, 7, 2, 4, 4, 5] OK
[5, 0, 3, 4] OK


In [13]:
class SolutionTwoPass:
    def getDistances(self, arr: List[int]) -> List[int]:
        n = len(arr)
        ans = [0] * n
        cnt, sm = {}, {}
        for i, v in enumerate(arr):
            c, s = cnt.get(v, 0), sm.get(v, 0)
            ans[i] = c * i - s
            cnt[v] = c + 1; sm[v] = s + i
        cnt, sm = {}, {}
        for i in range(n - 1, -1, -1):
            v = arr[i]
            c, s = cnt.get(v, 0), sm.get(v, 0)
            ans[i] += s - c * i
            cnt[v] = c + 1; sm[v] = s + i
        return ans

print(SolutionTwoPass().getDistances([10, 5, 10, 10]))

[5, 0, 3, 4]


**Độ phức tạp cách 4:** O(n) thời gian, O(số giá trị phân biệt) bộ nhớ phụ (ngoài mảng kết quả). Vì `arr[i] ≤ 10⁵`, có thể thay `dict` bằng list độ dài `10⁵ + 1` để nhanh hơn.

## 8. TRACE từng bước

### 8.1 Trace cách 4 trên ví dụ 1 `arr = [2,1,3,1,2,3,3]`

Lượt trái→phải (đóng góp bên trái):

| i | v | cnt[v] trước | sm[v] trước | left = cnt·i − sm |
|---|---|---|---|---|
| 0 | 2 | 0 | 0 | 0 |
| 1 | 1 | 0 | 0 | 0 |
| 2 | 3 | 0 | 0 | 0 |
| 3 | 1 | 1 | 1 | 3 − 1 = 2 |
| 4 | 2 | 1 | 0 | 4 − 0 = 4 |
| 5 | 3 | 1 | 2 | 5 − 2 = 3 |
| 6 | 3 | 2 | 7 | 12 − 7 = 5 |

Lượt phải→trái (đóng góp bên phải): `right = [4, 2, 7, 0, 0, 1, 0]`. Cộng lại: `[4, 2, 7, 2, 4, 4, 5]`. ✔

In [14]:
def trace_two_pass(arr):
    n = len(arr); left = [0] * n; right = [0] * n
    cnt, sm = defaultdict(int), defaultdict(int)
    print("Lượt trái -> phải")
    for i, v in enumerate(arr):
        left[i] = cnt[v] * i - sm[v]
        print(f"  i={i} v={v}: cnt={cnt[v]} sm={sm[v]} -> left={left[i]}")
        cnt[v] += 1; sm[v] += i
    cnt.clear(); sm.clear()
    print("Lượt phải -> trái")
    for i in range(n - 1, -1, -1):
        v = arr[i]
        right[i] = sm[v] - cnt[v] * i
        print(f"  i={i} v={v}: cnt={cnt[v]} sm={sm[v]} -> right={right[i]}")
        cnt[v] += 1; sm[v] += i
    ans = [a + b for a, b in zip(left, right)]
    print("left  =", left); print("right =", right); print("ans   =", ans)
    return ans

trace_two_pass([2, 1, 3, 1, 2, 3, 3])

Lượt trái -> phải
  i=0 v=2: cnt=0 sm=0 -> left=0
  i=1 v=1: cnt=0 sm=0 -> left=0
  i=2 v=3: cnt=0 sm=0 -> left=0
  i=3 v=1: cnt=1 sm=1 -> left=2
  i=4 v=2: cnt=1 sm=0 -> left=4
  i=5 v=3: cnt=1 sm=2 -> left=3
  i=6 v=3: cnt=2 sm=7 -> left=5
Lượt phải -> trái
  i=6 v=3: cnt=0 sm=0 -> right=0
  i=5 v=3: cnt=1 sm=6 -> right=1
  i=4 v=2: cnt=0 sm=0 -> right=0
  i=3 v=1: cnt=0 sm=0 -> right=0
  i=2 v=3: cnt=2 sm=11 -> right=7
  i=1 v=1: cnt=1 sm=3 -> right=2
  i=0 v=2: cnt=1 sm=4 -> right=4
left  = [0, 0, 0, 2, 4, 3, 5]
right = [4, 2, 7, 0, 0, 1, 0]
ans   = [4, 2, 7, 2, 4, 4, 5]


[4, 2, 7, 2, 4, 4, 5]

### 8.2 Trace cách 2 (tiền tố) cho nhóm giá trị 10 ở ví dụ 2

`p = [0, 2, 3]`, `pre = [0, 0, 2, 5]`, `k = 3`.

In [15]:
def trace_prefix_group(p):
    k = len(p); pre = list(accumulate(p, initial=0))
    print("p =", p, " pre =", pre)
    for j, pj in enumerate(p):
        left = pj * j - pre[j]; right = (pre[k] - pre[j + 1]) - pj * (k - j - 1)
        print(f"  j={j} p_j={pj}: trái = {pj}*{j} - {pre[j]} = {left};"
              f" phải = ({pre[k]} - {pre[j+1]}) - {pj}*{k-j-1} = {right}; S = {left + right}")

trace_prefix_group([0, 2, 3])

p = [0, 2, 3]  pre = [0, 0, 2, 5]
  j=0 p_j=0: trái = 0*0 - 0 = 0; phải = (5 - 0) - 0*2 = 5; S = 5
  j=1 p_j=2: trái = 2*1 - 0 = 2; phải = (5 - 2) - 2*1 = 1; S = 3
  j=2 p_j=3: trái = 3*2 - 2 = 4; phải = (5 - 5) - 3*0 = 0; S = 4


## 9. Code nộp LeetCode

Chọn cách 4 (hai lượt quét) — ngắn, O(n), không cần gom nhóm.

In [16]:
from typing import List
from collections import defaultdict

class Solution:
    def getDistances(self, arr: List[int]) -> List[int]:
        n = len(arr)
        ans = [0] * n
        cnt = defaultdict(int)   # số lần đã gặp giá trị v
        sm = defaultdict(int)    # tổng chỉ số các lần đã gặp v
        for i, v in enumerate(arr):            # các bạn bên trái
            ans[i] = cnt[v] * i - sm[v]
            cnt[v] += 1
            sm[v] += i
        cnt.clear(); sm.clear()
        for i in range(n - 1, -1, -1):         # các bạn bên phải
            v = arr[i]
            ans[i] += sm[v] - cnt[v] * i
            cnt[v] += 1
            sm[v] += i
        return ans

In [17]:
sol = Solution()
for arr, exp in examples:
    got = sol.getDistances(arr)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

[4, 2, 7, 2, 4, 4, 5] | kỳ vọng [4, 2, 7, 2, 4, 4, 5] | OK
[5, 0, 3, 4] | kỳ vọng [5, 0, 3, 4] | OK


## 10. Vì sao thuật toán đúng

Cố định `i` với giá trị `v`. Tập `J = {j : arr[j] = v, j ≠ i}` chia thành `J_L = {j < i}` và `J_R = {j > i}`.

`intervals[i] = Σ_{j∈J_L} (i − j) + Σ_{j∈J_R} (j − i) = (|J_L|·i − Σ J_L) + (Σ J_R − |J_R|·i)`.

**Lượt trái→phải:** bất biến *ngay trước khi xử lý `i`*, `cnt[v] = |{j < i : arr[j] = v}| = |J_L|` và `sm[v] = Σ J_L`. Bất biến đúng lúc đầu (rỗng), và sau khi xử lý `i` ta thêm đúng chỉ số `i` vào thống kê của `arr[i]` ⇒ vẫn đúng cho `i + 1`. Vậy phần thêm vào `ans[i]` chính là tổng bên trái.

**Lượt phải→trái:** đối xứng, `cnt[v] = |J_R|`, `sm[v] = Σ J_R` ⇒ cộng thêm tổng bên phải.

Hai phần rời nhau và phủ hết `J` ⇒ đáp án đúng. Cách 2 và 3 là cùng đẳng thức, chỉ tổ chức theo nhóm; cách 3 thu được bằng cách lấy hiệu `S_j − S_{j−1}` của công thức cách 2:
`S_j − S_{j−1} = j·Δ − (k − j)·Δ`.

In [18]:
# Kiểm chứng đẳng thức truy hồi bằng công thức tiền tố trên một nhóm ngẫu nhiên
rng = random.Random(10)
p = sorted(rng.sample(range(100), 8)); k = len(p); pre = list(accumulate(p, initial=0))
S = [p[j] * j - pre[j] + (pre[k] - pre[j + 1]) - p[j] * (k - j - 1) for j in range(k)]
for j in range(1, k):
    d = p[j] - p[j - 1]
    assert S[j] - S[j - 1] == j * d - (k - j) * d
print("p =", p); print("S =", S); print("đẳng thức truy hồi đúng")

p = [1, 4, 26, 54, 59, 61, 62, 73]
S = [332, 314, 226, 170, 170, 174, 178, 244]
đẳng thức truy hồi đúng


## 11. Phân tích Time / Space Complexity

`n = len(arr)`, `k_v` = số lần xuất hiện của `v` (`Σ k_v = n`).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute mọi cặp | O(n²) | O(1) |
| 1b. Cặp trong nhóm | O(Σ k_v²), worst O(n²) | O(n) |
| 2. Nhóm + tiền tố | O(n) | O(n) |
| 3. Nhóm + truy hồi | O(n) | O(n) |
| 4. Hai lượt quét | O(n) | O(số giá trị phân biệt) |

Đo trên `n = 10⁵`:

In [19]:
rng = random.Random(1)
tests = {"ngẫu nhiên 1..100": [rng.randint(1, 100) for _ in range(10**5)],
         "tất cả bằng nhau": [7] * 10**5}
for name, a in tests.items():
    res = []
    for fn in [group_prefix, group_rolling, two_pass, Solution().getDistances]:
        t = time.perf_counter(); r = fn(a); res.append((r, time.perf_counter() - t))
    assert all(r == res[0][0] for r, _ in res)
    print(name, "->", ", ".join(f"{dt*1000:.0f} ms" for _, dt in res))

ngẫu nhiên 1..100 -> 28 ms, 22 ms, 30 ms, 29 ms


tất cả bằng nhau -> 25 ms, 14 ms, 30 ms, 29 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {"brute": brute, "group_pairs": group_pairs, "group_prefix": group_prefix,
       "group_rolling": group_rolling, "two_pass": two_pass,
       "SolutionPrefix": SolutionPrefix().getDistances,
       "SolutionTwoPass": SolutionTwoPass().getDistances, "Solution": Solution().getDistances}
edge = examples + [
    ([1], [0]),
    ([5, 5], [1, 1]),
    ([1, 2, 3, 4], [0, 0, 0, 0]),
    ([4, 4, 4, 4], [6, 4, 4, 6]),
    ([1, 2, 1, 2, 1], [6, 2, 4, 2, 6]),
    ([100000, 1, 100000], [2, 0, 2]),
]
for arr, exp in edge:
    for name, f in ALL.items():
        got = f(list(arr))
        assert got == exp, (name, arr, got, exp)
print("Edge case OK:", len(edge))

Edge case OK: 8


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [21]:
rng = random.Random(2121)
for _ in range(2000):
    n = rng.randint(1, 40)
    hi = rng.choice([1, 2, 3, 10, 100000])
    arr = [rng.randint(1, hi) for _ in range(n)]
    ref = brute(arr)
    for name, f in ALL.items():
        a = list(arr)
        assert f(a) == ref, (name, arr)
        assert a == arr, (name, "đã sửa input")
print("2000 case ngẫu nhiên OK")

2000 case ngẫu nhiên OK


### 12.3 Kiểm tra tính chất: tổng đáp án = 2 × tổng khoảng cách mọi cặp cùng giá trị

In [22]:
rng = random.Random(77)
for _ in range(200):
    arr = [rng.randint(1, 5) for _ in range(rng.randint(1, 300))]
    pair_sum = sum(j - i for i in range(len(arr)) for j in range(i + 1, len(arr)) if arr[i] == arr[j])
    assert sum(Solution().getDistances(arr)) == 2 * pair_sum
print("tính chất tổng OK")

tính chất tổng OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên dấu: dùng `sm[v] − cnt[v]·i` ở lượt trái→phải

In [23]:
def wrong_sign(arr):
    n = len(arr); ans = [0] * n
    cnt, sm = defaultdict(int), defaultdict(int)
    for i, v in enumerate(arr):
        ans[i] += sm[v] - cnt[v] * i          # SAI dấu: các j bên trái nhỏ hơn i
        cnt[v] += 1; sm[v] += i
    cnt.clear(); sm.clear()
    for i in range(n - 1, -1, -1):
        v = arr[i]; ans[i] += sm[v] - cnt[v] * i
        cnt[v] += 1; sm[v] += i
    return ans

print("bản sai :", wrong_sign([2, 1, 3, 1, 2, 3, 3]))
print("bản đúng:", Solution().getDistances([2, 1, 3, 1, 2, 3, 3]))

bản sai : [4, 2, 7, -2, -4, -2, -5]
bản đúng: [4, 2, 7, 2, 4, 4, 5]


**Vì sao sai:** bên trái thì `j < i` nên `|i − j| = i − j` ⇒ tổng là `cnt·i − sm`. Đảo dấu cho ra số âm. **Sửa:** viết rõ `i − j` rồi mới cộng dồn. Mẹo kiểm tra: đáp án không bao giờ âm.

### Lỗi 2 — Cập nhật `cnt`, `sm` **trước** khi tính (tự tính chính mình)

In [24]:
def wrong_update_order(arr):
    n = len(arr); ans = [0] * n
    cnt, sm = defaultdict(int), defaultdict(int)
    for i, v in enumerate(arr):
        cnt[v] += 1; sm[v] += i               # SAI: cập nhật trước
        ans[i] += cnt[v] * i - sm[v]
    cnt.clear(); sm.clear()
    for i in range(n - 1, -1, -1):
        v = arr[i]; cnt[v] += 1; sm[v] += i
        ans[i] += sm[v] - cnt[v] * i
    return ans

print("bản sai :", wrong_update_order([2, 1, 3, 1, 2, 3, 3]))
print("bản đúng:", Solution().getDistances([2, 1, 3, 1, 2, 3, 3]))

bản sai : [4, 2, 7, 2, 4, 4, 5]
bản đúng: [4, 2, 7, 2, 4, 4, 5]


Hừm — kết quả **giống nhau**! Vì chính `i` đóng góp `i − i = 0`. Đây là ví dụ hay: thứ tự cập nhật ở *bài này* không gây sai, nhưng ở các biến thể (ví dụ đếm **số** phần tử bằng nhau bên trái, hoặc tổng `|i − j| + 1`) thì sai ngay. Xem phản ví dụ khi đếm cặp:

In [25]:
def count_left_equal_wrong(arr):
    cnt = defaultdict(int); out = []
    for v in arr:
        cnt[v] += 1; out.append(cnt[v])        # SAI: tính cả chính mình
    return out

def count_left_equal_right(arr):
    cnt = defaultdict(int); out = []
    for v in arr:
        out.append(cnt[v]); cnt[v] += 1
    return out

print("số phần tử bằng nhau đứng TRƯỚC (sai):", count_left_equal_wrong([2, 1, 2, 2]))
print("số phần tử bằng nhau đứng TRƯỚC (đúng):", count_left_equal_right([2, 1, 2, 2]))

số phần tử bằng nhau đứng TRƯỚC (sai): [1, 1, 2, 3]
số phần tử bằng nhau đứng TRƯỚC (đúng): [0, 0, 1, 2]


**Bài học:** luôn theo khuôn "**dùng thống kê → rồi mới cập nhật**" để thống kê luôn mô tả "các phần tử *trước* `i`".

### Lỗi 3 — Công thức truy hồi dùng `(k − j − 1)` thay vì `(k − j)`

In [26]:
def wrong_rolling(arr):
    groups = defaultdict(list)
    for i, v in enumerate(arr): groups[v].append(i)
    ans = [0] * len(arr)
    for p in groups.values():
        k = len(p); S = sum(p) - k * p[0]; ans[p[0]] = S
        for j in range(1, k):
            d = p[j] - p[j - 1]
            S += j * d - (k - j - 1) * d        # SAI: quên p_j cũng "gần lại"
            ans[p[j]] = S
    return ans

print("bản sai :", wrong_rolling([2, 1, 3, 1, 2, 3, 3]))
print("bản đúng:", group_rolling([2, 1, 3, 1, 2, 3, 3]))

bản sai : [4, 2, 7, 4, 8, 7, 9]
bản đúng: [4, 2, 7, 2, 4, 4, 5]


**Vì sao sai:** khi dịch từ `p_{j−1}` sang `p_j`, chính điểm `p_j` cũng "gần lại" `Δ` (từ khoảng cách `Δ` xuống 0). Các điểm gần lại là `p_j, …, p_{k−1}` — có `k − j` điểm. Các điểm xa thêm là `p_0, …, p_{j−1}` — `j` điểm (gồm cả `p_{j−1}`, từ 0 lên `Δ`). **Sửa:** `S += j·Δ − (k − j)·Δ`.

### Lỗi 4 — Gom nhóm bằng `arr.index(v)` / tìm kiếm tuyến tính lặp lại

In [27]:
def slow_index(arr):
    ans = []
    for i, v in enumerate(arr):
        idxs = [j for j in range(len(arr)) if arr[j] == v]     # O(n) mỗi lần -> O(n^2)
        ans.append(sum(abs(i - j) for j in idxs))
    return ans

a = [rng.randint(1, 3) for _ in range(3000)]
t = time.perf_counter(); r1 = slow_index(a); t1 = time.perf_counter() - t
t = time.perf_counter(); r2 = Solution().getDistances(a); t2 = time.perf_counter() - t
print(f"kết quả giống nhau: {r1 == r2}; chậm: {t1*1000:.0f} ms vs nhanh: {t2*1000:.1f} ms")

kết quả giống nhau: True; chậm: 343 ms vs nhanh: 0.8 ms


**Vì sao "sai":** đúng kết quả nhưng TLE với `n = 10⁵`. **Sửa:** gom nhóm **một lần** bằng `defaultdict(list)`, hoặc dùng hai lượt quét.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng `|i − j|` với mọi `j` thỏa điều kiện" ⇒ **tách trái/phải**, dùng **(đếm, tổng)** hoặc tổng tiền tố.
- Pattern "cnt·x − sum" xuất hiện ở: LC 1685 Sum of Absolute Differences in a Sorted Array, LC 2615 Sum of Distances, LC 2602 Minimum Operations to Make All Array Elements Equal, LC 462 (median).
- Khi điều kiện là "cùng giá trị" ⇒ thống kê theo **từng giá trị** bằng `dict`.
- Luôn kiểm tra nhanh: đáp án không âm; phần tử duy nhất ⇒ 0; tổng đáp án = 2 × tổng mọi cặp.

## 15. Biến thể của bài

### 15.1 Biến thể A — Tổng khoảng cách tới **mọi** phần tử (không cần cùng giá trị), mảng đã sắp xếp (LC 1685)

`result[i] = Σ_j |nums[i] − nums[j]|` với `nums` tăng dần — cùng công thức tiền tố nhưng trên **giá trị** thay vì chỉ số.

In [28]:
def sum_abs_diff_sorted(nums):
    n = len(nums); total = sum(nums); run = 0; out = []
    for i, x in enumerate(nums):
        out.append(x * i - run + (total - run - x) - x * (n - i - 1))
        run += x
    return out

print(sum_abs_diff_sorted([2, 3, 5]))        # [4, 3, 5]
print(sum_abs_diff_sorted([1, 4, 6, 8, 10])) # [24, 15, 13, 15, 21]

[4, 3, 5]
[24, 15, 13, 15, 21]


### 15.2 Biến thể B — Chỉ tính khoảng cách tới các phần tử cùng giá trị trong **cửa sổ** `|i − j| ≤ w`

Trong mỗi nhóm `p`, các `j` hợp lệ là một đoạn liên tiếp `[lo, hi]` tìm bằng `bisect` ⇒ vẫn dùng tiền tố. O(n log n).

In [29]:
from bisect import bisect_left, bisect_right
def distances_window(arr, w):
    groups = defaultdict(list)
    for i, v in enumerate(arr): groups[v].append(i)
    ans = [0] * len(arr)
    for p in groups.values():
        pre = list(accumulate(p, initial=0))
        for j, pj in enumerate(p):
            lo = bisect_left(p, pj - w); hi = bisect_right(p, pj + w)
            left = pj * (j - lo) - (pre[j] - pre[lo])
            right = (pre[hi] - pre[j + 1]) - pj * (hi - j - 1)
            ans[pj] = left + right
    return ans

def distances_window_brute(arr, w):
    return [sum(abs(i - j) for j in range(len(arr)) if arr[j] == arr[i] and abs(i - j) <= w) for i in range(len(arr))]

print(distances_window([2, 1, 3, 1, 2, 3, 3], 3))
rng = random.Random(152)
for _ in range(500):
    arr = [rng.randint(1, 3) for _ in range(rng.randint(1, 30))]; w = rng.randint(0, 10)
    assert distances_window(arr, w) == distances_window_brute(arr, w)
print("biến thể cửa sổ: khớp brute force")

[0, 2, 3, 2, 0, 4, 1]
biến thể cửa sổ: khớp brute force


### 15.3 Biến thể C — Tổng **bình phương** khoảng cách `Σ (i − j)²` tới phần tử cùng giá trị

Khai triển `(i − j)² = i² − 2ij + j²` ⇒ cần thống kê `cnt`, `Σj`, `Σj²` cho mỗi giá trị — không cần tách trái/phải vì bình phương không có dấu!

In [30]:
def squared_distances(arr):
    cnt, s1, s2 = defaultdict(int), defaultdict(int), defaultdict(int)
    for i, v in enumerate(arr):
        cnt[v] += 1; s1[v] += i; s2[v] += i * i
    return [cnt[v] * i * i - 2 * i * s1[v] + s2[v] for i, v in enumerate(arr)]

arr = [2, 1, 3, 1, 2, 3, 3]
print(squared_distances(arr))
assert squared_distances(arr) == [sum((i - j) ** 2 for j in range(len(arr)) if arr[j] == arr[i]) for i in range(len(arr))]

[16, 4, 25, 4, 16, 10, 17]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute mọi cặp | so mọi `(i, j)` | O(n²) | O(1) | kiểm chứng, n nhỏ |
| 1b. Cặp trong nhóm | chỉ so trong nhóm | O(Σk²) | O(n) | giá trị rất đa dạng |
| 2. Nhóm + tiền tố | công thức trái/phải | O(n) | O(n) | dễ mở rộng (cửa sổ, truy vấn) |
| 3. Nhóm + truy hồi | `S_j = S_{j−1} + (2j − k)Δ` | O(n) | O(n) | gọn, một biến |
| 4. Hai lượt quét | `(cnt, sum)` theo giá trị | O(n) | O(#giá trị) | **mặc định**, ngắn nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `getDistances([3, 3, 3])` và `getDistances([1, 2, 1])` là gì?

**Bài 2 (tính tay):** Với nhóm chỉ số `p = [1, 4, 5, 9]`, tính `S_0` rồi dùng truy hồi tính `S_1, S_2, S_3`.

**Bài 3 (điền TODO):** Viết `bt3_todo(arr)` trả về mảng `left` — chỉ phần tổng khoảng cách tới các phần tử **cùng giá trị đứng trước**.

**Bài 4 (tự viết lại):** Viết `bt4_todo(arr)` cài cách 2 (gom nhóm + tiền tố) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(arr)` trả về **chỉ số `i`** có `intervals[i]` nhỏ nhất **trong số các chỉ số có ít nhất một bạn cùng giá trị**; hòa thì lấy chỉ số nhỏ nhất; nếu mọi giá trị đều xuất hiện một lần thì trả −1.

In [31]:
def bt3_todo(arr):
    # TODO: tổng khoảng cách tới các phần tử cùng giá trị đứng TRƯỚC
    pass

def bt4_todo(arr):
    # TODO: gom nhóm + tổng tiền tố
    pass

def bt5_todo(arr):
    # TODO: argmin intervals trên các chỉ số có bạn cùng giá trị, hoặc -1
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tính trực tiếp. Mức 2: `[3,3,3]`: chỉ số 0 có bạn ở 1, 2. Mức 3: `[3, 2, 3]` và `[2, 0, 2]`.

**Bài 2.** Mức 1: `S_0 = Σp − 4·p_0`. Mức 2: `S_0 = 19 − 4 = 15`. Mức 3: `Δ = 3, 1, 4` ⇒ `S_1 = 15 + (2·1−4)·3 = 9`, `S_2 = 9 + (2·2−4)·1 = 9`, `S_3 = 9 + (2·3−4)·4 = 17`.

**Bài 3.** Mức 1: chỉ cần lượt trái→phải của cách 4. Mức 2: `cnt`, `sm` theo giá trị. Mức 3: `left[i] = cnt[v]*i − sm[v]` rồi cập nhật.

**Bài 4.** Mức 1: `defaultdict(list)`. Mức 2: `pre = accumulate(p, initial=0)`. Mức 3: `S_j = p_j·j − pre[j] + pre[k] − pre[j+1] − p_j·(k−j−1)`.

**Bài 5.** Mức 1: tính `intervals` trước. Mức 2: "có bạn" ⇔ giá trị xuất hiện ≥ 2 lần (dùng `Counter`). Mức 3: duyệt `i` tăng dần, chỉ xét `cnt[arr[i]] ≥ 2`, cập nhật khi nhỏ hơn hẳn.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[3, 2, 3]`, `[2, 0, 2]`. **Bài 2:** `S = [15, 9, 9, 17]`.

In [32]:
from collections import Counter
assert Solution().getDistances([3, 3, 3]) == [3, 2, 3]
assert Solution().getDistances([1, 2, 1]) == [2, 0, 2]
p = [1, 4, 5, 9]
assert [sum(abs(a - b) for b in p) for a in p] == [15, 9, 9, 17]

def bt3(arr):
    cnt, sm = defaultdict(int), defaultdict(int); left = []
    for i, v in enumerate(arr):
        left.append(cnt[v] * i - sm[v]); cnt[v] += 1; sm[v] += i
    return left

def bt4(arr):
    groups = defaultdict(list)
    for i, v in enumerate(arr): groups[v].append(i)
    ans = [0] * len(arr)
    for p in groups.values():
        k = len(p); pre = list(accumulate(p, initial=0))
        for j, pj in enumerate(p):
            ans[pj] = pj * j - pre[j] + (pre[k] - pre[j + 1]) - pj * (k - j - 1)
    return ans

def bt5(arr):
    iv = Solution().getDistances(arr); c = Counter(arr); best = -1
    for i, v in enumerate(arr):
        if c[v] >= 2 and (best == -1 or iv[i] < iv[best]):
            best = i
    return best

assert bt3([2, 1, 3, 1, 2, 3, 3]) == [0, 0, 0, 2, 4, 3, 5]
assert bt5([2, 1, 3, 1, 2, 3, 3]) == 1 and bt5([1, 2, 3]) == -1 and bt5([5, 5]) == 0
rng = random.Random(1921)
for _ in range(1500):
    arr = [rng.randint(1, 4) for _ in range(rng.randint(1, 30))]
    ref = brute(arr)
    assert bt4(arr) == ref
    assert bt3(arr) == [sum(i - j for j in range(i) if arr[j] == arr[i]) for i in range(len(arr))]
    cand = [i for i in range(len(arr)) if arr.count(arr[i]) >= 2]
    assert bt5(arr) == (min(cand, key=lambda i: (ref[i], i)) if cand else -1)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Tổng |i - j| tới các j cùng giá trị: hai lượt (cnt, sum)
ans = [0] * n
cnt, sm = defaultdict(int), defaultdict(int)
for i, v in enumerate(arr):              # trái
    ans[i] += cnt[v] * i - sm[v]; cnt[v] += 1; sm[v] += i
cnt.clear(); sm.clear()
for i in range(n - 1, -1, -1):           # phải
    v = arr[i]; ans[i] += sm[v] - cnt[v] * i; cnt[v] += 1; sm[v] += i
```

**Checklist:**
- [ ] Dấu đúng: trái `cnt·i − sum`, phải `sum − cnt·i`.
- [ ] Thứ tự: **dùng thống kê rồi mới cập nhật**.
- [ ] Reset `cnt`, `sm` giữa hai lượt.
- [ ] Phần tử xuất hiện một lần ⇒ 0; đáp án không âm.
- [ ] Không dùng tìm kiếm tuyến tính lặp lại (O(n²) TLE).